# HW9: images, trees, tours and geometric coverage
Download the course ZIP. Open this notebook in Colab and run the cells in order;
upload **HW9_data.zip** when prompted. No Drive mount or authentication is needed.
The ZIP provides a shuffled worked-example image and its correct row order,
helpers, point files and small illustrations. No original image is included:
reconstruct it yourself using the supplied order.
The ZIP also contains **HW9_rows_challenge_shuffled.png**, the assessed mystery.
No original, row order or shuffle seed is supplied for that image.

These helpers do not implement row ordering, MST, NN tours, DFS or weighted greedy.
Add your algorithms in new cells. A library MST routine is allowed.
Locally, install numpy, matplotlib, Pillow and Jupyter in your environment.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import json
import sys

archive_path = Path("HW9_data.zip")
if not archive_path.exists():
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError("Place HW9_data.zip beside this notebook.")
    files.upload()
if not archive_path.exists():
    raise FileNotFoundError("Upload the file named HW9_data.zip.")
data_dir = Path("hw9_data")
data_dir.mkdir(exist_ok=True)


In [ ]:
required = ['HW9_helpers.py', 'HW9_experiment_helpers.py', 'HW9_TSP_README.txt', 'HW9_TSP_manifest.json', 'HW9_small_examples.json', 'HW9_small_tours.png', 'HW9_small_cover.png', 'HW9_rows_test_shuffled.png', 'HW9_rows_test_order.txt', 'HW9_rows_test_info.json', 'HW9_rows_challenge_shuffled.png', 'HW9_TSP_baseline_tour.txt', 'HW9_TSP_baseline_score.json', 'HW9_TSP_100.txt', 'HW9_TSP_100.png', 'HW9_TSP_100.svg', 'HW9_TSP_500.txt', 'HW9_TSP_500.png', 'HW9_TSP_500.svg', 'HW9_TSP_2000.txt', 'HW9_TSP_2000.png', 'HW9_TSP_2000.svg']
with ZipFile(archive_path) as archive:
    if sum(info.file_size for info in archive.infolist()) > 50_000_000:
        raise ValueError("Unexpectedly large homework bundle")
    for name in required:
        # Exact known basenames only: never extract arbitrary ZIP paths.
        (data_dir / name).write_bytes(archive.read(name))
sys.path.insert(0, str(data_dir.resolve()))
import numpy as np
import matplotlib.pyplot as plt
import HW9_helpers as H
import HW9_experiment_helpers as E
%matplotlib inline
print("Data ready:", data_dir.resolve())


## Worked example: shuffled image only
Row indices start at **0**, top to bottom. Complete rows were randomly
permuted; their pixels were not changed, resized or shuffled within a row.
Do not resize the mystery image vertically: that would blend unrelated shuffled
rows. If creating a new exercise, resize the ORIGINAL before shuffling.
This example comes with a correct reordering. The challenge does NOT.


In [ ]:
shuffled = E.load_rgb(data_dir / "HW9_rows_test_shuffled.png")
E.show_rows([shuffled], ["Worked example: shuffled rows"])
plt.show()
print("Image shape (height, width, RGB channels):", shuffled.shape)


## Reconstruct the original using the supplied row order
Each line of **HW9_rows_test_order.txt** is a row index in the SHUFFLED image.
Read the list from top to bottom to obtain the original:

    original[y] = shuffled[correct_order[y]]

In NumPy this is simply **original = shuffled[correct_order]**.
The helper also checks that the order uses every row exactly once. The hash
verifies the restored RGB pixels without shipping a separate original PNG.
This is a worked example of applying a known order, NOT an unscrambling algorithm.


In [ ]:
import hashlib

correct_order = np.loadtxt(data_dir / "HW9_rows_test_order.txt", dtype=int)
info = json.loads((data_dir / "HW9_rows_test_info.json").read_text())
original = E.reconstruct_rows(shuffled, correct_order)
assert hashlib.sha256(original.tobytes()).hexdigest() == info["original_rgb_sha256"]
E.save_rgb(original, "HW9_example_restored.png")
print("First ten shuffled row indices to take:", correct_order[:10].tolist())
print("Verified: reconstructed pixels match the original exactly.")
E.show_rows([original], ["Reconstructed from the supplied order"])
plt.show()


## Now try discovering an order yourself
For practice, ignore the answer list and use row distances to construct your
own order. Evaluate it against the known order afterwards. This calculation
uses ALL rows of the full-resolution example and may take a few seconds.
The mystery image has no correct-order file.

Hint: an image has a top and a bottom. Grow a greedy NN chain at both ends:
compare the best unused-row match for prepending and for appending, then make
the better extension. Never connect the last row back to the first. Break
ties by row index, then prefer prepending if both ends tie. You may experiment
with other similarity measures; the helper supplies absolute RGB difference.
Compare numerical path scores only within the same measure.


In [ ]:
row_distances = E.row_distance_matrix(shuffled)
print("Distance matrix:", row_distances.shape)
print("First 5 x 5 distances:")
print(row_distances[:5, :5])


## Your proposed reconstruction
Supply a permutation of shuffled row indices. These calls assemble and score
it; they do not choose a good order. The score is an OPEN path, with no closing edge.

    my_order = your_row_ordering(row_distances)
    reconstructed = E.reconstruct_rows(shuffled, my_order)
    print(E.row_path_cost(row_distances, my_order))
    # The inverse maps shuffled indices back to their original row indices.
    p = np.argsort(correct_order)
    print(E.recovered_adjacencies(my_order, p))  # worked example only
    E.save_rgb(reconstructed, "my_reconstruction.png")
    E.show_rows([shuffled, reconstructed], ["Shuffled", "My reconstruction"])

To create another testing pair:
    another_shuffled, another_p = E.shuffle_rows(original, seed=42)

Keep the answer list out of your search. It is only for assembling the worked
example above and checking the order found by your algorithm afterwards.


## Challenge: discover the order, then identify the image
This is a different image with a different height. Its complete rows have
been permuted independently, with no resizing or changes within rows.
Use your algorithm; the example's answer list does not apply.


In [ ]:
challenge = E.load_rgb(data_dir / "HW9_rows_challenge_shuffled.png")
E.show_rows([challenge], ["Challenge: what is in this image?"])
plt.show()
print("Challenge shape:", challenge.shape)
challenge_distances = E.row_distance_matrix(challenge)
print("Challenge distance matrix:", challenge_distances.shape)


Try at least three starting rows. Save your best zero-based row order, score
and reconstructed PNG, for example in your own cell:

    my_order = your_row_ordering(challenge_distances)
    restored = E.reconstruct_rows(challenge, my_order)
    print(E.row_path_cost(challenge_distances, my_order))
    np.savetxt("my_challenge_order.txt", my_order, fmt="%d")
    E.save_rgb(restored, "my_challenge_reconstruction.png")

There is no known-order adjacency score for the challenge. Interpret the
recovered image, even if some row joins are wrong. Browser/notebook previews
are scaled for viewing: use the downloaded PNG data, not a screenshot.


## Shared 100 / 500 / 2000 point inputs
Point IDs start at **1**, unlike Python array indices. Use points[id - 1].
Squared distances are useful for neighbour comparisons and radii; tour length
uses ordinary Euclidean distances. Keep squared and ordinary units distinct.
T3 tests EVERY start, not a sample: 2N fixed-MST traversal tours (two child
orders per root) and N NN tours. Reuse distances, the MST and, optionally,
sorted neighbour lists. Report best/worst starts and routes, median, spread
and total all-start runtime; keep preparation time separate.


In [ ]:
datasets = {n: H.load_points(data_dir / f"HW9_TSP_{n}.txt") for n in (100, 500, 2000)}
points = datasets[100]
d2 = E.squared_distances(points)
E.draw_geometry(points, title="100 input points")
plt.show()
print("Distance matrix shape:", d2.shape)
print("Squared distance between IDs 1 and 2:", d2[0, 1])


## Small worked examples: same MST, different tours
The stored 12-point tree and tours are illustrations, not solutions to the
100 / 500 / 2000 point tasks. The plotting helper accepts YOUR edge list or
tour in the same formats. It does not calculate a tree or tour.


In [ ]:
example = json.loads((data_dir / "HW9_small_examples.json").read_text())
small = example["points"]
participant_name = "Donald Duck"  # Replace with your own name for submissions.
fig, axes = plt.subplots(2, 2, figsize=(11, 10), layout="constrained")
E.draw_geometry(small, edges=example["mst_edges"], ax=axes[0, 0], labels=True,
                title="The fixed MST")
for ax, key, title in zip(axes.flat[1:],
    ["mst_tour_root1_ascending", "mst_tour_root8_descending", "nn_tour_root1"],
    ["MST: root 1, ascending", "MST: root 8, descending", "NN: start 1"]):
    tour = example[key]
    E.draw_geometry(small, tour=tour, ax=ax, labels=True, title=title,
                    name=participant_name)
plt.show()


## Save and validate your tour
The closing edge is implicit: each ID appears once. Each tour image requires
your name and automatically displays the independently recomputed closed-tour
length (three decimals) and the drawing's UTC date/time. The full-precision
validator JSON, not the rounded image caption, decides record improvements.
The timestamp is the image creation time, not a Zulip submission timestamp.

This runnable example uses the small worked tour, not an assessed solution:


In [ ]:
demo_tour = example["nn_tour_root1"]
H.draw_png(small, "Donald_Duck_example_tour.png", tour=demo_tour, name=participant_name)
H.draw_svg(small, "Donald_Duck_example_tour.svg", tour=demo_tour, name=participant_name)
print(H.tour_caption(small, demo_tour, participant_name))


For your own route:

    length = H.validate_tour(points, my_tour)
    Path("my_tour.txt").write_text("\n".join(map(str, my_tour)) + "\n")
    H.draw_png(points, "my_tour.png", tour=my_tour, name=participant_name)
    E.draw_geometry(points, tour=my_tour, name=participant_name)
    plt.savefig("my_tour_matplotlib.png", dpi=160, bbox_inches="tight")
    E.draw_geometry(points, edges=my_mst, title="My MST")
    plt.savefig("my_mst.png", dpi=160, bbox_inches="tight")

For competition claims, run the CLI with --competition to enforce the frozen
2000-point input hash. The supplied baseline route/score are for comparison,
not a substitute for your T2 implementation.


## Disk membership, cost and first-cover colours
K includes the centre. YOU find its K-1 nearest other points and choose the
radius squared. The helper includes every boundary tie and calculates the
full area. Distinct candidates are identified by (centre_id, radius_squared).
This small illustration uses fewer than ten members; assessed candidates do not.


In [ ]:
one_disk = E.disk_from_radius(small, centre_id=1, radius_squared=340**2)
print("Members:", one_disk["members"], "area:", round(one_disk["area"], 2))
E.draw_cover(small, [one_disk], min_members=1, title="One illustrative candidate; uncovered points are crosses")
plt.show()


For your chosen candidate radii:

    candidate = E.disk_from_radius(points, centre_id, radius_squared)
    # Your greedy algorithm selects a list of these disk dictionaries.
    result = E.check_cover(points, selected_disks)
    assert result["valid"], result["uncovered"]
    print(result["selected_count"], result["total_area"])
    E.draw_cover(points, selected_disks)
    plt.savefig("my_cover.png", dpi=160, bbox_inches="tight")

First-cover colours are a display convention: overlaps still belong to all
applicable disks and are charged repeatedly in the objective. For T5 and B1,
retain the complete candidate family and selected disk dictionaries as JSON.
Save the same points/candidates with both covers for a counterexample.
